# Preparing Arbitrary Quantum States Workbook

This workbook describes the solutions to the problems offered in the "Preparing Arbitrary Quantum States" kata. Since the tasks are offered as programming problems, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick, units
from math import atan2, sqrt

### Problem 1.1. Prepare arbitrary single-qubit state

As we have seen in the earlier katas, a qubit in the $\ket{0}$ state can be converted to a superposition of states with real coefficients using an $Ry$ gate: 

$$R_y(\theta) = \begin{bmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{bmatrix}$$

This gate turns the state $\ket{0}$ into $Ry(\theta)\ket{0} = \cos\frac{\theta}{2} \ket{0} + \sin\frac{\theta}{2} \ket{1}$,
which is similar to the state you need. You just need to find an angle $\theta$ such that $\cos\frac{\theta}{2}=\alpha$ and $\sin\frac{\theta}{2}=\beta$. You can use these two equations to solve for $\theta$ to get $\theta = 2\arctan\frac{\beta}{\alpha}$. (Remember that $\alpha^2 + \beta^2=1$.)

To make sure you don't need any special handling for negative or zero values, you can use the Python function `atan2`. Remember that its return is in radians, and the $Ry$ gate takes the rotation angle in degrees. This means that you need to convert the angle to degrees, for example, by multiplying it by `units.rad`.

In [ ]:
def prepare_one_qubit_state(reg: Qubits, alpha: float, beta: float) -> None:
    theta = 2 * atan2(beta, alpha)
    reg.ry(theta * units.rad)

### Problem 1.2. Prepare single-qubit state conditionally

The initial state of the register is $\ket{0}\ket{\psi} = \gamma\ket{00} + \delta\ket{01}$.

Let's consider the two scenarios for the different values of $c$ separately.

1. $c = 0$. 

   In this case, you need to transform the basis state $\ket{00}$ to $(\alpha\ket{0} + \beta\ket{1}) \otimes \ket{0}$ and leave the basis state $\ket{01}$ unchanged. In other words, you need to apply the $Ry$ gate from the previous problem to the first qubit using the second qubit as control, as long as you control the gate on the second qubit being in the $\ket{0}$ state.

2. $c = 1$.

    Similarly, you need to transform the basis state $\ket{01}$ to $(\alpha\ket{0} + \beta\ket{1}) \otimes \ket{1}$ and leave the basis state $\ket{00}$ unchanged, which you can do using the same $Ry$ gate, controlled on the second qubit being in the $\ket{1}$ state.

Both scenarios can be expressed in the same way: apply the $Ry$ gate controlled on the second qubit being in the state $\ket{c}$. In Workbench, you can express this control condition by passing `reg[1] == c` as the `cond` argument to the `ry` method.

In [ ]:
def prepare_conditional_state(reg: Qubits, alpha: float, beta: float, c: int) -> None:
    theta = 2 * atan2(beta, alpha)
    reg[0].ry(theta * units.rad, cond=reg[1] == c)

### Problem 1.3. Prepare superposition of three two-qubit basis states

You can represent the state we want to prepare as follows: 

$$a_0 \ket{00} + a_1 \ket{10} + a_2 \ket{01} = (a_0 \ket{0} + a_1 \ket{1}) \ket{0} + a_2 \ket{01}$$

$$ = \sqrt{a_0^2 + a_1^2} \left( \frac{a_0}{\sqrt{a_0^2 + a_1^2}} \ket{0} + \frac{a_1}{\sqrt{a_0^2 + a_1^2}} \ket{1} \right) \ket{0} + a_2 \ket{01}$$

This suggests a path to the solution:

1. Prepare the second (most significant) qubit in the state $\sqrt{a_0^2 + a_1^2} \ket{0} + a_2\ket{1}$. Same as in problem 1.1, we can do this using an $Ry$ gate with rotation angle $\theta = 2 \cdot \mathrm{atan2} \left( a_2, \sqrt{a_0^2 + a_1^2} \right)$.

2. Use controlled-on-zero $Ry$ gate to change the first term of the two-qubit state from $\sqrt{a_0^2 + a_1^2}\ket{0} \otimes \ket{0}$ to $a_0|00\rangle + a_1|10\rangle$. The rotation angle for this gate is $\theta_0 = 2 \cdot \mathrm{atan2} \left( \frac{a_1}{\sqrt{a_0^2 + a_1^2}}, \frac{a_0}{\sqrt{a_0^2 + a_1^2}} \right)$. Since $\mathrm{atan2}$ doesn't require its arguments to be normalized, we can just use $\theta_0 = 2 \cdot \mathrm{atan2}(a_1, a_0)$ instead.

In [ ]:
def prepare_three_basis_states_two_qubits(reg: Qubits, a: list[float]) -> None:
    b0 = sqrt(a[0] ** 2 + a[1] ** 2)
    b1 = a[2]
    # Prepare the most significant qubit
    reg[1].ry(2 * atan2(b1, b0) * units.rad)
    # Prepare the least significant qubit for MSB=0
    reg[0].ry(2 * atan2(a[1], a[0]) * units.rad, cond=reg[1] == 0)

### Problem 1.4. Prepare arbitrary two-qubit state

Similarly to the previous problem, you can represent the state we want to prepare as follows: 

$$a_0 \ket{00} + a_1 \ket{10} + a_2 \ket{01} + a_3 \ket{11} = (a_0 \ket{0} + a_1 \ket{1}) \ket{0} + (a_2 \ket{0} + a_3 \ket{1}) \ket{1}$$

$$ = \sqrt{a_0^2 + a_1^2} \left( \frac{a_0}{\sqrt{a_0^2 + a_1^2}} \ket{0} + \frac{a_1}{\sqrt{a_0^2 + a_1^2}} \ket{1} \right) \ket{0}
   + \sqrt{a_2^2 + a_3^2} \left( \frac{a_2}{\sqrt{a_2^2 + a_3^2}} \ket{0} + \frac{a_3}{\sqrt{a_2^2 + a_3^2}} \ket{1} \right) \ket{1}$$

The solution then follows the same principle as in problem 1.3, but with an extra step:

1. Prepare the second (most significant) qubit in the state $\sqrt{a_0^2 + a_1^2} \ket{0} + \sqrt{a_2^2 + a_3^2} \ket{1}$ using an $Ry$ gate.

2. Use controlled-on-zero $Ry$ gate to change the first term of the two-qubit state from $\sqrt{a_0^2 + a_1^2}\ket{0} \otimes \ket{0}$ to $a_0|00\rangle + a_1|10\rangle$.

3. Use controlled-on-one $Ry$ gate to change the second term of the two-qubit state from $\sqrt{a_2^2 + a_3^2}\ket{0} \otimes \ket{1}$ to $a_2|01\rangle + a_3|11\rangle$.

In [ ]:
def prepare_two_qubit_state(reg: Qubits, a: list[float]) -> None:
    b0 = sqrt(a[0] ** 2 + a[1] ** 2)
    b1 = sqrt(a[2] ** 2 + a[3] ** 2)
    # Prepare the most significant qubit
    reg[1].ry(2 * atan2(b1, b0) * units.rad)
    # Prepare the least significant qubit for MSB=0
    reg[0].ry(2 * atan2(a[1], a[0]) * units.rad, cond=reg[1] == 0)
    # Prepare the least significant qubit for MSB=0
    reg[0].ry(2 * atan2(a[3], a[2]) * units.rad, cond=reg[1] == 1)

### Problem 1.5. Prepare arbitrary multi-qubit state

If the number of qubits is $1$, we can just use the solution to problem 1.1 without modifications. 

For multiple qubits ($n \ge 2$), let's see how to modify the solution to the previous problem to cover this scenario using recursion.

First, we'll rewrite the state we want to prepare as a sum of two terms: those with the most significant bit (stored in the last qubit) equal to $0$ and those with it equal to $1$. We'll also separate the state of the last qubit from the states of the first $n-1$ qubits:

$$a_0 \ket{0} + a_1 \ket{1} + ... + a_{2^n - 1} \ket{2^n - 1} = (a_0 \ket{0} + ... + a_{2^{n-1} - 1} \ket{2^{n-1} - 1}) \otimes \ket{0} + (a_{2^{n-1}} \ket{0} + ... + a_{2^n - 1} \ket{2^{n-1} - 1}) \otimes \ket{1}$$

> Here the integers denoting the state of the $n-1$ least significant qubits are $n-1$-bit integers, and the most significant bit equal to $1$ encodes the additional value $2^{n-1}$. For each term, these two values add up to the index of the corresponding element of $a$.

This grouping emphasizes the recursive structure of the problem - the same one we used for the two-qubit case:

1. Prepare the most significant qubit in some superposition state.
2. Prepare $n-1$ least significant qubits in some superposition state, conditioned on the most significant qubit being $\ket{0}$.
3. Prepare $n-1$ least significant qubits in another superposition state, conditioned on the most significant qubit being $\ket{1}$.

To figure out the details, we can rewrite the target state as follows:

$$\frac1{\sqrt{a_0^2 + ... + a_{2^{n-1} - 1}^2}} (a_0 \ket{0} + ... + a_{2^{n-1} - 1} \ket{2^{n-1} - 1}) \otimes \sqrt{a_0^2 + ... + a_{2^{n-1} - 1}^2} \ket{0}$$

$$ + \frac1{\sqrt{a_{2^{n-1}}^2 + ... + a_{2^n - 1}^2}} (a_{2^{n-1}} \ket{0} + ... + a_{2^n - 1} \ket{2^{n-1} - 1}) \otimes \sqrt{a_{2^{n-1}}^2 + ... + a_{2^n - 1}^2} \ket{1}$$

This means that

1. On the first step, we prepare the most significant qubit in the state 

$$\sqrt{a_0^2 + ... + a_{2^{n-1} - 1}^2} \ket{0} + \sqrt{a_{2^{n-1}}^2 + ... + a_{2^n - 1}^2} \ket{1}$$

2. On the second step, we conditionally prepare the $n-1$ least significant qubits in the state

   $$a_0 \ket{0} + ... + a_{2^{n-1} - 1} \ket{2^{n-1} - 1}$$

   (We can ignore the normalization coefficient, since all our rotation angles are calculated using the function $\mathrm{atan2}$ which considers the ratio of its arguments rather than their values.)

3. On the third step, we conditionally prepare the $n-1$ least significant qubits in the state

   $$a_{2^{n-1}} \ket{0} + ... + a_{2^n - 1} \ket{2^{n-1} - 1}$$

The only new feature of the code implementation is using an additional argument `ctrl` in the `compute` method of our Qubrick. This argument specifies the control register and is optional: passing it indicates calling the controlled variant of the Qubrick. In our case, we expect the end-user code (such as the testing harness for this problem) to call the plain (uncontrolled) routine, but the implementation itself will be calling controlled variants of the routine.

Implementing the controlled version of the routine is straightforward. When using the $Ry$ gate to prepare the most significant qubit, we pass `ctrl` as the `cond` argument. When calling controlled state preparation for $n-1$ target qubits, we need to use a concatenation of the control register `ctrl` and any additional control qubits (in our case, the most significant qubit).


In [ ]:
class NaiveStatePrep(Qubrick):
    def _compute(self, reg: Qubits, a: list[float], ctrl: Qubits | None = None) -> None:
        """Prepare the register reg in the superposition of all basis states defined by amplitudes a."""
        n = len(reg)
        if n == 1:
            # Base case of recursion
            theta = 2 * atan2(a[1], a[0])
            reg.ry(theta * units.rad, cond=ctrl)
        else:
            # Separate all amplitudes into those with MSB = 0 and MSB = 1
            a_msb0 = a[:2 ** (n - 1)]
            a_msb1 = a[2 ** (n - 1):]
            b0 = sqrt(sum(aj * aj for aj in a_msb0))
            b1 = sqrt(sum(aj * aj for aj in a_msb1))
            
            # Prepare the most significant qubit
            self._compute(reg[-1], [b0, b1], ctrl)

            # Controlled-on-zero state prep on n-1 least significant qubits
            self._compute(reg[:-1], a_msb0, ctrl=ctrl | ~reg[-1])

            # Controlled-on-one state prep on n-1 least significant qubits
            self._compute(reg[:-1], a_msb1, ctrl=ctrl | reg[-1])

> You'll notice that we make a lot of assumptions in our code: we assume that the given qubit register is not empty, that the length of the amplitudes list matches the number of amplitudes required to describe the register of the given length, that the amplitudes list is not all zeroes... We're omitting the checks for these scenarios to keep this kata focused on the core algorithmic principles. When developing a real code library, you should definitely include these kinds of checks!

> Copyright (c) 2026 PsiQuantum